# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [102]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from urllib.parse import urljoin, urlparse
from IPython.display import Markdown, display, update_display
#from scraper import fetch_website_links, fetch_website_contents
import scraper
import importlib
from openai import OpenAI

In [76]:
# Initialize and constants

'''load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-5-nano'
openai = OpenAI() '''

OLLAMA_BASE_URL = "http://localhost:11434/v1"
ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')
MODEL = "llama3.2"

In [103]:
importlib.reload(scraper)
#links = scraper.fetch_website_links("https://edwarddonner.com")
links = scraper.fetch_website_links("https://huggingface.co")
links

['https://huggingface.co/',
 'https://huggingface.co/models',
 'https://huggingface.co/datasets',
 'https://huggingface.co/spaces',
 'https://huggingface.co/storage',
 'https://huggingface.co/docs',
 'https://huggingface.co/enterprise',
 'https://huggingface.co/pricing',
 'https://huggingface.co/tasks',
 'https://huggingface.co/chat',
 'https://huggingface.co/collections',
 'https://huggingface.co/languages',
 'https://huggingface.co/organizations',
 'https://huggingface.co/blog',
 'https://huggingface.co/posts',
 'https://huggingface.co/papers',
 'https://huggingface.co/hardware',
 'https://huggingface.co/learn',
 'https://huggingface.co/join/discord',
 'https://discuss.huggingface.co/',
 'https://github.com/huggingface',
 'https://huggingface.co/pro',
 'https://huggingface.co/support',
 'https://huggingface.co/inference/models',
 'https://huggingface.co/inference-endpoints',
 'https://huggingface.co/login',
 'https://huggingface.co/join',
 'https://huggingface.co/thinkingmachines/Ink

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [92]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, Careers/Jobs pages, or even blog.

You should respond in strict JSON structure as the following:
{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://full.url/careers"}
    ]
}
"""

In [93]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links. Do not invent new links.

Links (some might be relative links):

"""
    links = scraper.fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [94]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links. Do not invent new links.

Links (some might be relative links):

https://edwarddonner.com/avatar/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/
https://edwardd

In [95]:
def select_relevant_links(url):
    response = ollama.chat.completions.create(
        model="llama3.2",
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [96]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'Company page', 'url': 'https://edwarddonner.com/'},
  {'type': 'Careers/Jobs page', 'url': ''},
  {'type': 'Blog posts', 'url': 'https://edwarddonner.com/posts/'},
  {'type': "LinkedIn profile ( employer's perspective)",
   'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'Contact/Newsletter link from LinkedIn', 'url': ''}]}

In [97]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = ollama.chat.completions.create(
        model= MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [98]:
select_relevant_links("https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling llama3.2
Found 2 relevant links


{'links': [{'type': 'about me page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'LinkedIn profile',
   'url': 'https://www.linkedin.com/in/eddonner/'}]}

In [99]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling llama3.2
Found 7 relevant links


{'links': [{'type': 'abouts page', 'url': 'https://huggingface.co/about'},
  {'type': 'company page', 'url': 'https://huggingface.co/about'},
  {'type': 'model hub ', 'url': 'https://huggingface.co/models'},
  {'type': 'dataset hub', 'url': 'https://huggingface.co/datasets'},
  {'type': 'co-founder', 'url': 'https://about.huggingface.co/team/'},
  {'type': 'alliance', 'url': 'https://bit.ly/3rDcAqW'},
  {'type': 'hardware', 'url': 'https://huggingface.co/hardware'}]}

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [106]:
def fetch_page_and_all_relevant_links(url):
    result = scraper.fetch_website_contents(url)
    relevant_links = select_relevant_links(url)

    for link in relevant_links.get("links", []):
        link_type = link.get("type", "Unknown")
        link_url = link.get("url")

        if not isinstance(link_url, str):
            print(f"Skipping invalid link: {link}")
            continue

        link_url = link_url.strip()

        if not link_url:
            print(f"Skipping empty URL: {link}")
            continue

        # Fragment-only links such as "#/models" are not standalone pages.
        if link_url.startswith("#"):
            print(f"Skipping fragment-only URL: {link_url}")
            continue

        # Convert relative links such as "/models" into absolute URLs.
        absolute_url = urljoin(url, link_url)

        parsed = urlparse(absolute_url)

        if parsed.scheme not in {"http", "https"} or not parsed.netloc:
            print(f"Skipping malformed URL: {absolute_url}")
            continue

        result += f"\n\n### Link: {link_type}\n"
        result += scraper.fetch_website_contents(absolute_url)

    return result

In [107]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling llama3.2
Found 16 relevant links
Skipping empty URL: {'type': 'about us', 'url': ''}
Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Buckets
new
Docs
Enterprise
Pricing
Website
Tasks
HuggingChat
Collections
Languages
Organizations
Community
Blog
Posts
Daily Papers
Hardware
Learn
Discord
Forum
GitHub
Solutions
Team & Enterprise
Hugging Face PRO
Enterprise Support
Inference Providers
Inference Endpoints
Storage Buckets
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
thinkingmachines/Inkling
Updated
1 day ago
•
7.87k
•
916
prism-ml/Ternary-Bonsai-27B-gguf
Updated
5 minutes ago
•
201k
•
657
prism-ml/Bonsai-27B-gguf
Updated
7 minutes ago
•
1.05M
•
377
empero-ai/Qwythos-9B-Claude-Mythos-5-1M-GGUF
Updated
3 days ago
•
2

In [ ]:
'''brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
""" '''

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""


In [109]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [110]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling llama3.2
Found 2 relevant links


'\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nBuckets\nnew\nDocs\nEnterprise\nPricing\nWebsite\nTasks\nHuggingChat\nCollections\nLanguages\nOrganizations\nCommunity\nBlog\nPosts\nDaily Papers\nHardware\nLearn\nDiscord\nForum\nGitHub\nSolutions\nTeam & Enterprise\nHugging Face PRO\nEnterprise Support\nInference Providers\nInference Endpoints\nStorage Buckets\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\nthinkingmachines/Inkling\nUpdated\n1 day ago\n•\n7.87k\n•\n916\nprism-ml/Ternary-Bonsai-27B-gguf\nUpdated\n8 minutes ago\n•\n201k\n•\n65

In [111]:
def create_brochure(company_name, url):
    response = ollama.chat.completions.create(
        model = MODEL,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [112]:
create_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling llama3.2
Found 6 relevant links


# Hugging Face: Building the Future of AI Together

At Hugging Face, we are passionate about advancing the field of artificial intelligence through collaboration and innovation. Our mission is to create a platform where the machine learning community can come together to share knowledge, resources, and ideas.

## Collaboration is at the Heart

Our platform allows users to host, discover, and collaborate on unlimited public models, datasets, and applications. This open approach empowers individuals from all over the world to contribute their expertise and participate in the development of cutting-edge AI solutions.

### Features and Capabilities

* **2M+ Models and 1M+ Applications**: Explore a vast library of high-quality models and applications built by our community.
* **Run Machine Learning Workloads on WebGPU**: Take advantage of powerful web-based GPU computing with our Bonsai 27B kernels.
* **Reproduce Papers with Your Agent**: Utilize our capabilities to reproduce ICML papers and accelerate research.

## A Community of Experts

Our platform is supported by a vibrant community of researchers, developers, and practitioners working together to advance AI research and innovation.

### Current Openings:

Check out the current job openings at Hugging Face. We're looking for talented professionals to join our team!

[Learn More: Career/Jobs](#career-jobs)

## Join Our Community

Stay up-to-date with the latest developments, best practices, and community discussions through our forum and blog.

*   [Hugging Face Forums - Official Discussion Platform](#hugging-face-forums)
*   [Visit our Blog for Insights and Interviews](#our-blog)

### Connect with Us

Join us on various social media channels to stay informed about the latest breakthroughs and updates from Hugging Face.

[Learn More: Social Media Links](#social-media-links)

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [113]:
def stream_brochure(company_name, url):
    stream = ollama.chat.completions.create(
        model = MODEL,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [114]:
stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling llama3.2
Found 6 relevant links


**Welcome to Hugging Face: The AI Community Building the Future**

Hugging Face is a leading platform in the machine learning community, united by a shared passion for advancing artificial intelligence. Our mission is to create a collaborative space where developers, researchers, and users can come together to build, share, and apply AI models.

**What We Do**

At Hugging Face, we offer a vast ecosystem of tools and resources for building, deploying, and using AI models. Our flagship model platforms are designed to help users create, discover, and collaborate on machine learning projects. Our comprehensive platform includes:

* 2 million+ pre-trained models and applications
* 500 thousand+ datasets for machine learning
* Unlimeted public models, datasets, and applications for collaboration
* Inference providers and endpoints for deploying AI models

**Community-Driven**

Hugging Face is built by the community, for the community. We believe that knowledge sharing, open-source collaboration, and community engagement are essential to accelerating progress in AI research. Our platform enables users to:

* Host and share their own models and datasets
* Engage with other developers and researchers through our forums and social channels
* Contribute to open-source projects and advance the field of machine learning

**Supporting Innovation**

At Hugging Face, we're committed to helping innovators and businesses leverage AI to solve real-world problems. Our platform offers solutions for:

* NLP modeling and application development
* Data science and machine learning toolkits
* Inference and deployment services for AI models

**Join the Movement**

Hugging Face is a community of passionate individuals who share a common goal: building the future of artificial intelligence. Whether you're a researcher, developer, or entrepreneur, we invite you to join our ecosystem.

**Careers at Hugging Face**

We offer career opportunities in cutting-edge fields like AI research, software development, and data science. Our talented team is pushing the boundaries of what's possible with machine learning. Join us!

**Partnerships and Enterprise Solutions**

Hugging Face offers tailored solutions for enterprises and businesses looking to integrate our platform into their operations. Get in touch with us to explore how we can help you unlock the full potential of your AI investment.

Visit us today at [www.huggingface.co](http://www.huggingface.co)

In [ ]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>